In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import glob

## Analysis 1

In [ ]:
# Specify important meta information
out_parent_dir = '/gfriedri/hubo/Dp_manifold/juvenile_dataset/results/capacity'
os.makedirs(out_parent_dir, exist_ok=True)
windows = [[77, 101], [85, 108]]
# [[32, 85]]#[[38, 76]] # [[38, 63]] #[[40, 63]]#[[17, 41], [25, 48], [70, 93]]#[[32, 85]]#[[36, 74], [38, 76]]#[38, 68] #[[36, 70]]#[[36, 66], [38, 68]]#[[40, 70], [47, 78], [55, 85], [62, 93], [70, 100]]
seed = 2345

from catrace.dataset import load_dataset_config
config_file = f'../../../notebooks/juvenile_dataset_without14mm_phearg.json'
dsconfig = load_dataset_config(config_file)
from os.path import join as pjoin
# Define your jobs
jobs_all = []

indir = dsconfig.processed_trace_dir
exp_list = dsconfig.exp_list

Ns = [700] #[200, 300, 400, 500, 600]#, 700]
M = 70 #140#100
all_odors = dsconfig.odors_stimuli

import sys

# Import from current directory
from catrace.run.run_capacity import AnalysisParams
params = AnalysisParams(
    outfile='',
    indir=indir,
    fish_id=None,
    condition=None,
    window=None,
    odors=None,
    N=Ns[0],
    M=M,
    seed=None,
    overwrite_computation=True,
    gaussianize=False,
)
params

AnalysisParams(outfile='', indir='/tungstenfs/scratch/gfriedri/hubo/Dp_manifold/juvenile_dataset/data/spike_prob_long/per_odor_aligned/index_formatted_firing_rate', fish_id=None, condition=None, window=None, N=700, M=70, seed=None, n_hyperplanes=1000, shuffle=True, analysis_type='FIRST_VERSUS_REST', scale=1, geometry=True, gaussianize=False, odors=None, overwrite_computation=True, debug=False)

In [4]:
import os
from dataclasses import replace

def get_odor_pairs(all_odors):
    # Get all pairs of odors, including (A, B) and (B, A)
    odor_pairs = []
    for i in range(len(all_odors)):
        for j in range(len(all_odors)):
            if i != j:
                odor_pairs.append((all_odors[i], all_odors[j]))
    return odor_pairs

def deduplicate_unordered_pairs(pairs):
    unordered_pairs = list(set(tuple(sorted(p)) for p in pairs))
    return unordered_pairs


import os
from os.path import join as pjoin
from itertools import product
import multiprocessing as mp
    # Define a worker function
def process_window(args):
    rep, fish_id, condition, odor_pair, window, seed, N, master_seed = args

    rep_dir = f'repeat_{rep:03d}'
    odor_pair_dir = f'odor_{odor_pair[0]}_{odor_pair[1]}'
    fish_id_dir = f'{fish_id}'
    window_tag = f'window_{window[0]}_{window[1]}'
    param_subdir = f'capacity_sweep_{window_tag}_N{N}_M{M}_master_seed{master_seed}'
    outdir  = os.path.join(out_parent_dir, param_subdir)
    #out_dir = os.path.join(outdir, rep_dir, odor_pair_dir, fish_id_dir)
    #os.makedirs(out_dir, exist_ok=True)
    os.makedirs(outdir, exist_ok=True)
    file_name = f'{rep_dir}_{odor_pair_dir}_{fish_id_dir}_{window_tag}.pkl'
    outfile = os.path.join(outdir, file_name)

    job = replace(params, outfile=outfile,
                    window=window,
                    fish_id=fish_id, condition=condition,
                    seed=seed, odors=odor_pair, N=N).to_dict()
    return job

def generate_jobs_mp(rep, exp_list, all_odors, seed,
                     windows, Ns):
    odor_pairs = get_odor_pairs(all_odors)
    unordered_odor_pairs = deduplicate_unordered_pairs(odor_pairs)

    # Update seed
    master_seed = seed
    base_seed = seed + rep * len(exp_list) * len(odor_pairs)

    # Prepare arguments for parallel processing
    tasks = []
    idx = 0  # Index to increment seed
    for window in windows:
        for N in Ns:
            for fish_id, condition in exp_list:
                for odor_pair in unordered_odor_pairs:
                    tasks.append((rep, fish_id, condition, odor_pair, window, base_seed + idx, N, master_seed))
                    idx += 1

    # Use multiprocessing Pool to process windows in parallel
    with mp.Pool() as pool:
        jobs_all = pool.map(process_window, tasks)

    return jobs_all



import os
import glob
import json
import math

# Parameters
start_rep = 0
num_rep = 50
overwrite = True
job_dir = './jobs_sweep'

# New parameters
num_tasks = 80                 # Number of array tasks
num_processes_per_task = 24   # Number of processes per task
# num_runs_per_process will be calculated based on total jobs

# Generate all jobs
jobs_all = []
for rep in range(start_rep, start_rep + num_rep):
    jobs_rep = generate_jobs_mp(rep, exp_list, all_odors, seed,
                                windows, Ns)
    jobs_all.extend(jobs_rep)

total_runs = len(jobs_all)
total_processes = num_tasks * num_processes_per_task

# Calculate runs per process
num_runs_per_process = total_runs // total_processes
leftover_runs = total_runs % total_processes

# Initialize the tasks structure
tasks = [ [ [] for _ in range(num_processes_per_task) ] for _ in range(num_tasks) ]

# Distribute jobs among tasks and processes
job_index = 0
for process_id in range(total_processes):
    task_id = process_id // num_processes_per_task
    process_rank = process_id % num_processes_per_task

    # Determine number of runs for this process
    runs_to_assign = num_runs_per_process + (1 if process_id < leftover_runs else 0)

    # Assign runs to the process
    runs = jobs_all[job_index : job_index + runs_to_assign]
    tasks[task_id][process_rank] = runs
    job_index += runs_to_assign

import concurrent.futures
def write_task_json(args):
    print('Writing task', args[0])
    task_id, processes = args
    output_file = os.path.join(job_dir, f'task_{task_id}.json')
    with open(output_file, 'w') as f:
        json.dump(processes, f, indent=4)

# Write each task's jobs to a JSON file
if not os.path.exists(job_dir) or overwrite:
    os.makedirs(job_dir, exist_ok=True)
    # Remove existing job files
    for f in glob.glob(os.path.join(job_dir, '*.json')):
        os.remove(f)

    tasks_to_write = list(enumerate(tasks))
    with concurrent.futures.ThreadPoolExecutor() as executor:
        executor.map(write_task_json, tasks_to_write)


Writing taskWriting task 1
 0
Writing task 2
Writing task 3
Writing task 4
Writing task 5
Writing task 6
Writing task 7
Writing task 8
Writing task 9
Writing task 10
Writing task 11
Writing task 12
Writing task 13
Writing task 14
Writing task 15
Writing task 16
Writing task 17
Writing task 18
Writing task 19
Writing task 20
Writing task 21
Writing task 22
Writing task 23
Writing task 24
Writing task 25
Writing task 26
Writing task 27
Writing task 28
Writing task 29
Writing task 30
Writing task 31
Writing task 32
Writing task 33
Writing task 34
Writing task 35
Writing task 36
Writing task 37
Writing task 38
Writing task 39
Writing task 40
Writing task 41
Writing task 42
Writing task 43
Writing task 44
Writing task 45
Writing task 46
Writing task 47
Writing task 48
Writing task 49
Writing task 50
Writing task 51
Writing task 52
Writing task 53
Writing task 54
Writing task 55
Writing task 56
Writing task 57
Writing task 58
Writing task 59
Writing task 60
Writing task 61
Writing task 62
Wr

In [6]:
import numpy as np
# Print the sbatch command
print(f'sbatch --ntasks-per-node={num_processes_per_task} --array=0-{num_tasks-1} slurm_job_sweep.sh')

sbatch --ntasks-per-node=24 --array=0-79 slurm_job_sweep.sh


In [7]:
# Save the first job to the current directory for testing
test_job = jobs_all[0]
test_job['overwrite_computation'] = True
test_job['debug'] = True
with open('test_job.json', 'w') as f:
    json.dump(test_job, f, indent=4)